# Import

In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from skipkv.monkeypatch import replace_qwen2, replace_qwen2_steering

import random
import numpy as np

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.cuda.manual_seed_all(seed)

set_seed(42)


# Prompt Template

prompt_template = "You are given a math problem.\n\nProblem: {question}\n\n You need to solve the problem step by step. First, you need to provide the chain-of-thought, then provide the final answer.\n\n Provide the final answer in the format: Final answer:  \\boxed{{}}"

In [ ]:
prompt_template = "You are given a math problem.\n\nProblem: {question}\n\n You need to solve the problem step by step. First, you need to provide the chain-of-thought, then provide the final answer.\n\n Provide the final answer in the format: Final answer:  \\boxed{{}}"

dataset2key = {
    "gsm8k": ["question", "answer"],
    "aime24": ["question", "answer"],
    "math": ["problem", "answer"],
}

prompts = []
test_data = []
questions_json = []

with open("../data/math.jsonl", "r") as f:
    for index, line in enumerate(f):
        example = json.loads(line)
        question_key = dataset2key['math'][0]

        question = example[question_key]
        example["question"] = question
        prompt = prompt_template.format(**example)
        questions_json.append(example)

        example["prompt"] = prompt
        example["index"] = index
        prompts.append(prompt)
        test_data.append(example)

from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
tokenizer = AutoTokenizer.from_pretrained(
    model_name, use_fast=True, padding_side="left"
)

SEAL=True

if SEAL:
    prefix="Answer the following questions. You should think step-by-step and put your final answer within \\boxed{}.\n"
    prompts = []
    for i, example in enumerate(test_data):
        prompt =  prefix+"Question: " + example["question"].strip()+"\nAnswer: "
        messages = [{"role": "user", "content": prefix + "Question: " + example["question"].strip()}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt = prompt[len(tokenizer.bos_token):]
        prompts.append(prompt)

prompts = prompts[:100]
test_data = test_data[:100]

device = torch.device("cuda:0")

# Rank batches by length for multi-batch decoding (shortest → longest)
prefill_lengths = []
for p in prompts:
    tp = tokenizer(
        p,
        return_tensors="pt",
        add_special_tokens=True,
    ).to(device)
    prefill_len = tp["attention_mask"].sum(dim=1).item()
    prefill_lengths.append(prefill_len)
order = sorted(range(len(prefill_lengths)), key=lambda i: prefill_lengths[i])
prompts = [prompts[i] for i in order]
test_data = [test_data[i] for i in order]


# Select Model

Choose from:
- deepseek-ai/DeepSeek-R1-Distill-Llama-8B
- deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
- deepseek-ai/DeepSeek-R1-Distill-Qwen-14B
- deepseek-ai/DeepSeek-R1-Distill-Qwen-32B

In [ ]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
print(len(prompts))
print([len(p) for p in prompts])

# Method Configuration

- Choose method from: skipkv, rkv, snapkv, streamingllm, h2o, analysiskv
- Choose budget from: 128, 256, 512, 1024
- Choose mix_lambda from 0 to 1.

- When mix_lambda=0, the selection is dominated by redundency.
- When mix_lambda=1, the selection is dominated by attention.

- Analysiskv is a special method that we do not compress KV but only return the selection patterns. In this way, we could observe the importance score without compressing KV.

In [ ]:
compression_config = {
    "method": "skipkv",
    "method_config": {
        "budget": 1024,
        "window_size": 8,
        "mix_lambda": 0.1,
        "retain_ratio": 0.2,
        "S_threshold": 0.99,
        "retain_direction": "last",
        "record_kept_token_indices": True,
    },
    "compression": None,
    "update_kv": True
}

model_config = {
    "divide_method": "step_length",
    "divide_length": 128,
    "compression_content": "all",
}

## Load Model

In [ ]:
import os
from huggingface_hub import login

# Use local_files_only=True to avoid redownloading
tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    use_fast=True, 
    padding_side="left",
    local_files_only=True,  # Use only cached files
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

replace_qwen2(compression_config)

if SEAL:
    replace_qwen2_steering()

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    local_files_only=True,  # Use only cached files
    attn_implementation='flash_attention_2',
).eval()

model.config.update(model_config)

model.newline_token_ids = [
    tokenizer.encode("\n")[-1],
    tokenizer.encode(".\n")[-1],
    tokenizer.encode(")\n")[-1],
    tokenizer.encode("\n\n")[-1],
    tokenizer.encode(".\n\n")[-1],
    tokenizer.encode(")\n\n")[-1],
    tokenizer.encode("?\n\n")[-1],
]

model.after_think_token_ids = [
    tokenizer.encode("</think>")[-1],
]

if SEAL:
    steering_vector = './SEAL/results/MATH_train/DeepSeek-R1-Distill-Qwen-7B/baseline_10000/vector_500_500/layer_20_transition_reflection_steervec.pt'
    vector_name_split = steering_vector.split("/")[-3:]
    vector_name_split[-1] = vector_name_split[-1].split(".")[0]
    name = "_".join(vector_name_split)
    steer_vec = torch.load(steering_vector, weights_only=True)
    steer_vec = steer_vec.to(model.device)
    model.set_steering_flag(steering_flag=True, steering_layer=20, steer_vec=steer_vec,  steer_coef=-1.0, tokenizer=tokenizer)

model.to(device)



In [ ]:
wait_tokens = ["Alternatively", "Wait", "again"]
wait_token_ids = [tokenizer.encode(t)[-1] for t in wait_tokens]

# Get punctuation token IDs individually
newline_token_ids = ["\n", ".\n", ")\n", "\n\n", ".\n\n", ")\n\n", "?\n\n"]
model.newline_token_ids = [tokenizer.encode(t)[-1] for t in newline_token_ids]

if compression_config['method'].lower() == "skipkv":
    model.enable_wait_token_monitoring(wait_token_ids, model.newline_token_ids, tokenizer=tokenizer)


# Inference

In [ ]:
if compression_config['method'].lower() == "skipkv":
    model.reset_for_new_sample()
    
if SEAL:
    model.start_new_round(-1.0)

# inputs = tokenizer(prompt, return_tensors="pt").to(device)
from tqdm import tqdm

sample_id = 1
inputs = {}
for i in tqdm(range(0, 10, 10)):
    batch_prompts = prompts[i : i + 10]
    tokenized_prompts = tokenizer(
        batch_prompts,
        padding="longest",
        return_tensors="pt",
        add_special_tokens=True,
    ).to("cuda")
    
    question = prompts[sample_id]
    prompt = tokenized_prompts['input_ids'][sample_id]
    mask = tokenized_prompts['attention_mask'][sample_id]
    
    sample_id_org = order[sample_id]
    sample = test_data[sample_id_org]
    answer = sample["answer"].split("####")[-1].strip()
    print('prompt: ', prompt)
    print('answer: ', answer)

    inputs['input_ids'] = prompt.unsqueeze(0).to(device)
    inputs['attention_mask'] = mask.unsqueeze(0).to(device)

    outputs = model.generate(
        **inputs,
        max_length=8192,
        num_beams=1,
        do_sample=False,
        use_cache=True
    )

    ## Display model answer
    print("Sample ID: ", sample_id_org)
    print(
        tokenizer.decode(outputs[0], skip_special_tokens=True),
    )
    print("\n\nGround Truth: ", answer)
    print("Generation length:", len(outputs[0]) - inputs['input_ids'].shape[1])
    print(
        "Compression Steps:",
        len(model.model.layers[-1].self_attn.kv_cluster.kept_token_indices),
    )
    print("Evicted tokens:", model.model.layers[-1].self_attn.kv_cluster.evicted_token_num)
    torch.cuda.empty_cache()

# Visualize token eviction

## Visualize the token eviction pattern for a given head for one compression step

In [ ]:
from skipkv.utils import visualize_token_eviction

layer_id = 27
head_id = 0

kept_indices_lst = model.model.layers[layer_id].self_attn.kv_cluster.kept_token_indices
visualize_token_eviction(
    outputs[0], kept_indices_lst, tokenizer, head_idx=head_id, kv_budget=1024, save_path='./skipkv.html'
)

# print(model.model.layers[0].self_attn.config.num_key_value_heads)

## Visualize the token eviction pattern for a given heads at each compression step

In [ ]:
from skipkv.utils import visualize_multistep_token_eviction

layer_id = 27
head_id = 0
step_idx = 4


kept_indices_lst = model.model.layers[layer_id].self_attn.kv_cluster.kept_token_indices
visualize_multistep_token_eviction(
    outputs[0], kept_indices_lst, tokenizer, head_idx=head_id, step_idx=step_idx
)

## Visualize the token eviction pattern for all heads at each compression step

In [ ]:
from skipkv.utils import visualize_multistep_token_eviction_by_head

layer_id = 27
step_idx = -1

kept_indices_lst = model.model.layers[layer_id].self_attn.kv_cluster.kept_token_indices

print("Total Step: ", len(kept_indices_lst))
visualize_multistep_token_eviction_by_head(
    outputs[0], kept_indices_lst, tokenizer, step_idx=step_idx, aggregate=True, save_path='./skipkv_heads.html'
)

# aggregate: when set to False, later heads will cover previous heads. when set to `True`, will compute how many times a token are covered by a head.

## Visualize the token eviction score for all heads at each compression step

In [ ]:
from skipkv.utils import visualize_multistep_token_eviction_score_by_head

layer_id = 27
head_idx = 3
step_idx = -1

kept_indices_lst = model.model.layers[layer_id].self_attn.kv_cluster.kept_token_indices
kept_attention_scores_lst = model.model.layers[layer_id].self_attn.kv_cluster.kept_attention_scores

visualize_multistep_token_eviction_score_by_head(
    outputs[0], kept_indices_lst, kept_attention_scores_lst, tokenizer, step_idx=step_idx, head_idx=head_idx,
)